In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("cms_dpoa_data_final_clean.csv")

# Preview
display(df.head())
print(df.shape)
print(df.columns)

In [ ]:
# Clean useful numeric columns
numeric_cols = [
    "year_published",
    "events_total_num",
    "events_used_num",
    "collision_energy_tev_num",
    "luminosity_pb",
    "fraction_used"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Basic summary
summary = {
    "Number of rows": len(df),
    "Unique papers": df["paper"].nunique(),
    "Unique datasets": df["dataset_key"].nunique(),
    "Rows with event counts": df["events_used_num"].notna().sum(),
    "Rows with fraction used": df["fraction_used"].notna().sum()
}

pd.DataFrame(summary.items(), columns=["Metric", "Value"])

In [ ]:
# Count data vs simulation datasets
dataset_counts = df["dataset_type"].value_counts()

plt.figure(figsize=(7, 5))
dataset_counts.plot(kind="bar")
plt.title("CMS Open Data Usage by Dataset Type")
plt.xlabel("Dataset Type")
plt.ylabel("Number of Dataset Mentions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Number of papers by year
papers_by_year = df.drop_duplicates("paper").groupby("year_published").size()

plt.figure(figsize=(9, 5))
papers_by_year.plot(kind="bar")
plt.title("Number of Academic Papers Using CMS Open Data by Year")
plt.xlabel("Year Published")
plt.ylabel("Number of Papers")
plt.tight_layout()
plt.show()

In [ ]:
# Number of dataset mentions by year
mentions_by_year = df.groupby("year_published").size()

plt.figure(figsize=(9, 5))
mentions_by_year.plot(kind="bar")
plt.title("Number of CMS Dataset Mentions by Year")
plt.xlabel("Year Published")
plt.ylabel("Dataset Mentions")
plt.tight_layout()
plt.show()

In [ ]:
# Top datasets used
top_datasets = df["dataset_name"].value_counts().head(15)

plt.figure(figsize=(10, 6))
top_datasets.sort_values().plot(kind="barh")
plt.title("Most Frequently Mentioned CMS Open Data Datasets")
plt.xlabel("Number of Mentions")
plt.ylabel("Dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Fraction of available events used
fraction_df = df[df["fraction_used"].notna()].copy()

plt.figure(figsize=(8, 5))
plt.hist(fraction_df["fraction_used"], bins=20)
plt.title("Distribution of Fraction of Events Used")
plt.xlabel("Fraction Used")
plt.ylabel("Number of Dataset Mentions")
plt.tight_layout()
plt.show()

In [ ]:
# Events used vs total events
event_df = df[
    df["events_total_num"].notna() &
    df["events_used_num"].notna()
].copy()

plt.figure(figsize=(7, 6))
plt.scatter(event_df["events_total_num"], event_df["events_used_num"])
plt.xscale("log")
plt.yscale("log")
plt.title("Events Used vs. Total Events Available")
plt.xlabel("Total Events Available")
plt.ylabel("Events Used")
plt.tight_layout()
plt.show()

In [ ]:
# Average fraction used by dataset type
avg_fraction_by_type = (
    df[df["fraction_used"].notna()]
    .groupby("dataset_type")["fraction_used"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(7, 5))
avg_fraction_by_type.plot(kind="bar")
plt.title("Average Fraction of Events Used by Dataset Type")
plt.xlabel("Dataset Type")
plt.ylabel("Average Fraction Used")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Luminosity used by papers
lum_df = (
    df.drop_duplicates(["paper", "luminosity_pb"])
    .dropna(subset=["luminosity_pb"])
)

plt.figure(figsize=(8, 5))
plt.hist(lum_df["luminosity_pb"], bins=20)
plt.title("Distribution of Integrated Luminosity Used in Papers")
plt.xlabel("Luminosity Used (pb⁻¹)")
plt.ylabel("Number of Papers / Dataset Entries")
plt.tight_layout()
plt.show()

In [ ]:
# Top papers by number of datasets mentioned
datasets_per_paper = (
    df.groupby("shortened_paper_name")["dataset_key"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
datasets_per_paper.sort_values().plot(kind="barh")
plt.title("Papers Mentioning the Most Distinct CMS Datasets")
plt.xlabel("Number of Distinct Datasets")
plt.ylabel("Paper")
plt.tight_layout()
plt.show()

In [ ]:
#Useful table :)
report_table = df.groupby("paper").agg(
    shortened_paper_name=("shortened_paper_name", "first"),
    year_published=("year_published", "first"),
    journal=("journal", "first"),
    num_datasets=("dataset_key", "nunique"),
    num_data_datasets=("dataset_type", lambda x: (x == "data").sum()),
    num_mc_datasets=("dataset_type", lambda x: (x == "mc").sum()),
    total_events_used=("events_used_num", "sum"),
    avg_fraction_used=("fraction_used", "mean"),
    max_luminosity_pb=("luminosity_pb", "max")
).reset_index()

report_table.sort_values("num_datasets", ascending=False).head(20)

In [ ]:
# Save cleaned summary table
report_table.to_csv("cms_open_data_usage_summary_by_paper.csv", index=False)